# B2 · L2 MCTS+PUCT Reasoning Runtime

**PHYRE W3 Track B · Colab Pro+ A100 80GB · ~30–60 min**

## 目标
把 W1 的 PRM (`prm_v1.pt`, traj AUC 0.829) 接入 MCTS+PUCT 搜索，在 §9E.1 L2 子集（60 题）上做 step-by-step reasoning。

## 算法
- **State**: 当前 reasoning chain (有序 sub-rule 列表 + 起点问题)
- **Action**: 从候选 sub-rule 集合里挑下一步
- **PUCT**: `Q(s,a) + c_puct · P(s,a) · √Σ_b N(s,b) / (1 + N(s,a))`
- **Q**: PRM 在 partial chain 上的打分（训练时的 trajectory-level Sigmoid head 直接用）
- **Prior P**: DeepSeek-V3 propose 的 top-k action 给均匀概率（W4 替换为学得 policy）
- **Stop**: 达到 ground_truth chain 长度，或 PRM ≥ 0.9 连续 2 步

## 评测
- top-1 EM (chain 完全相同) ≥ 0.55
- 部分匹配 F1 (chunk overlap) ≥ 0.70
- 平均搜索宽度 ≤ 8 actions per node (cost budget)

In [ ]:
# ========== 1. setup ==========
import os, sys, json, math, time, random, warnings
from pathlib import Path
from dataclasses import dataclass, field
import numpy as np, torch
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PHYRE = Path('/content/drive/MyDrive/phyre')
except Exception:
    PHYRE = Path('phyre')

DATA  = PHYRE / 'data' / 'benchmark'
SRC   = PHYRE / 'src'
LOGS  = PHYRE / 'logs'; LOGS.mkdir(parents=True, exist_ok=True)
CKPT  = PHYRE / 'ckpt' / 'prm_v1.pt'
BENCH = DATA / 'echem_reason_benchmark.jsonl'

sys.path.insert(0, str(SRC))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0); random.seed(0)
print(f'device={device}  ckpt={CKPT.exists()}  bench={BENCH.exists()}')

# DeepSeek API key (for action proposer); skip if absent
DSK = os.environ.get('DEEPSEEK_API_KEY') or (PHYRE / '.deepseek_key').read_text().strip() if (PHYRE / '.deepseek_key').exists() else None
print(f'deepseek key configured: {DSK is not None}')

In [ ]:
# ========== 2. load benchmark + PRM ==========
with open(BENCH, encoding='utf-8') as f:
    BENCH_ALL = [json.loads(l) for l in f]
L2 = [e for e in BENCH_ALL if e.get('level') == 2]
print(f'§9E.1 L2 questions: {len(L2)}  (target: 60)')

if L2:
    gt0 = L2[0].get('ground_truth', {})
    print(f'  L2[0] qid={L2[0].get("qid")}  gt keys: {list(gt0.keys())}')
    if 'mechanism_subgraph' in gt0:
        sg = gt0['mechanism_subgraph']
        print(f'  L2[0] subgraph: {len(sg.get("nodes",[]))} nodes, {len(sg.get("edges",[]))} edges')

# ground-truth reasoning chain — derive from mechanism_subgraph (topo sort)
def _verbalize(node):
    V = (node.get('V') or '').strip()
    S = ' '.join(node.get('S', []) or [])
    M = ' '.join(node.get('M', []) or [])
    C = ' '.join(node.get('C', []) or [])
    parts = [p for p in [V, S, M, C] if p]
    return ' '.join(parts).lower().strip()

def _toposort(nodes, edges):
    nid = {n['id']: n for n in nodes if 'id' in n}
    indeg = {i: 0 for i in nid}
    adj = {i: [] for i in nid}
    for e in edges or []:
        s, d = e.get('src'), e.get('dst')
        if s in nid and d in nid:
            adj[s].append(d); indeg[d] += 1
    order, q = [], [i for i,d in indeg.items() if d == 0]
    while q:
        u = q.pop(0); order.append(u)
        for v in adj[u]:
            indeg[v] -= 1
            if indeg[v] == 0: q.append(v)
    for i in nid:
        if i not in order: order.append(i)
    return [nid[i] for i in order]

def gt_chain(entry):
    gt = entry.get('ground_truth', {})
    for k in ('reasoning_chain', 'sub_rules', 'chain', 'steps'):
        v = gt.get(k)
        if isinstance(v, list) and v:
            if isinstance(v[0], dict):
                v = [c.get('rule') or c.get('text') or c.get('statement','') for c in v]
            return [str(c).strip().lower() for c in v if c]
    sg = gt.get('mechanism_subgraph') or {}
    nodes, edges = sg.get('nodes', []), sg.get('edges', [])
    if nodes:
        return [_verbalize(n) for n in _toposort(nodes, edges) if _verbalize(n)]
    ke = gt.get('key_elements') or []
    return [str(k).strip().lower() for k in ke if k]

lens = [len(gt_chain(e)) for e in L2]
if lens:
    print(f'chain length: min={min(lens)}  median={int(np.median(lens))}  max={max(lens)}')
    print(f'  example L2[0] chain: {gt_chain(L2[0])[:5]}')
else:
    print('chain length: N/A (empty L2)')

# load PRM — probe candidate paths (prefer newest version)
PRM_CANDIDATES = [
    PHYRE / 'ckpt' / 'prm_v2.pt',
    PHYRE / 'ckpt' / 'prm_v1.pt',
    PHYRE / 'ckpt' / 'prm.pt',
    PHYRE / 'ckpt' / 'prm_best.pt',
    PHYRE / 'pvgap_experiment' / 'ckpt' / 'prm_v2.pt',
    PHYRE / 'pvgap_experiment' / 'ckpt' / 'prm_v1.pt',
]
CKPT_FOUND = next((p for p in PRM_CANDIDATES if p.exists()), None)
if CKPT_FOUND is None and (PHYRE / 'ckpt').exists():
    cands = sorted((PHYRE / 'ckpt').glob('*prm*.pt'), key=lambda p: p.stat().st_mtime, reverse=True)
    CKPT_FOUND = cands[0] if cands else None
if CKPT_FOUND is None:
    print(f'⚠ no PRM ckpt found; tried {[str(p) for p in PRM_CANDIDATES]}')
    print('  → using untrained DeBERTa head (zero-shot). Rerun B1 to produce ckpt.')
else:
    print(f'PRM ckpt: {CKPT_FOUND}')

from transformers import AutoTokenizer, AutoModelForSequenceClassification
PRM_BACKBONE = 'microsoft/deberta-v3-large'
tok = AutoTokenizer.from_pretrained(PRM_BACKBONE)
prm = AutoModelForSequenceClassification.from_pretrained(PRM_BACKBONE, num_labels=1, ignore_mismatched_sizes=True)
if CKPT_FOUND is not None:
    try:
        state = torch.load(CKPT_FOUND, map_location=device, weights_only=False)
    except TypeError:
        state = torch.load(CKPT_FOUND, map_location=device)
    sd = state.get('model', state) if isinstance(state, dict) else state
    res = prm.load_state_dict(sd, strict=False)
    print(f'  loaded; missing={len(res.missing_keys)} unexpected={len(res.unexpected_keys)}')
prm.to(device).eval()
print(f'PRM ready, params={sum(p.numel() for p in prm.parameters())/1e6:.1f}M')

@torch.no_grad()
def prm_score(question, chain):
    """PRM trajectory-level score on (question, partial chain)."""
    text = question + ' [SEP] ' + ' [STEP] '.join(chain) if chain else question
    enc = tok(text, max_length=512, truncation=True, return_tensors='pt').to(device)
    logit = prm(**enc).logits.squeeze().item()
    return float(torch.sigmoid(torch.tensor(logit)).item())

In [ ]:
# ========== 3. action proposer (DeepSeek-V3) ==========
import urllib.request, urllib.error

def deepseek_propose(question, partial_chain, k=6, temperature=0.7):
    """Ask DeepSeek-V3 for k candidate next sub-rules. Returns list[str] (deduped, lowercased)."""
    if DSK is None:
        # offline stub: 6 generic action templates per node
        return [f'mechanism step {i+1} for {question[:40]}' for i in range(k)]
    sys_prompt = ('You are an expert in lithium-ion battery degradation reasoning. Given a question and a partial '
                  'reasoning chain (each step is a one-sentence sub-rule), propose K plausible NEXT sub-rules. '
                  'Return JSON array of strings, no markdown, no commentary.')
    user_prompt = (f'Question: {question}\n\nPartial chain so far ({len(partial_chain)} steps):\n'
                   + '\n'.join(f'  {i+1}. {s}' for i, s in enumerate(partial_chain))
                   + f'\n\nPropose K={k} candidate next sub-rules as JSON array.')
    body = json.dumps({
        'model': 'deepseek-chat',
        'messages': [{'role': 'system', 'content': sys_prompt}, {'role': 'user', 'content': user_prompt}],
        'temperature': temperature, 'max_tokens': 600,
        'response_format': {'type': 'json_object'},
    }).encode('utf-8')
    req = urllib.request.Request('https://api.deepseek.com/v1/chat/completions',
                                  data=body, headers={'Content-Type':'application/json',
                                                       'Authorization': f'Bearer {DSK}'})
    try:
        resp = urllib.request.urlopen(req, timeout=30)
        out = json.loads(resp.read())['choices'][0]['message']['content']
        arr = json.loads(out)
        if isinstance(arr, dict):
            arr = arr.get('sub_rules') or arr.get('next') or list(arr.values())[0]
        return [str(x).strip().lower() for x in arr][:k]
    except Exception as e:
        warnings.warn(f'deepseek_propose failed ({e}); using offline stub')
        return [f'mechanism step {i+1}' for i in range(k)]

# sanity
props = deepseek_propose(L2[0]['question_text'], gt_chain(L2[0])[:1], k=6)
for i, p in enumerate(props): print(f'  {i+1}. {p[:90]}')

In [ ]:
# ========== 4. MCTS+PUCT search ==========
@dataclass
class Node:
    chain: tuple = ()                            # path from root
    children: dict = field(default_factory=dict) # action_str -> Node
    N: int = 0                                   # visits
    W: float = 0.0                               # accumulated PRM
    prior_P: float = 0.0                         # uniform from proposer
    expanded: bool = False

    @property
    def Q(self): return self.W / max(self.N, 1)

C_PUCT = 1.5
MAX_DEPTH_FACTOR = 1.5      # max depth = ceil(MAX_DEPTH_FACTOR * len(gt_chain))
K_BRANCH = 6                # actions per node
N_SIMS = 24                 # MCTS simulations per question
PRM_HIGH = 0.9              # early-stop confidence

def select(node):
    """PUCT child selection."""
    sumN = sum(c.N for c in node.children.values())
    best, best_score = None, -1e18
    for a, c in node.children.items():
        score = c.Q + C_PUCT * c.prior_P * math.sqrt(sumN) / (1 + c.N)
        if score > best_score: best, best_score = a, score
    return best

def expand(node, question):
    actions = deepseek_propose(question, list(node.chain), k=K_BRANCH)
    p_uniform = 1.0 / max(len(actions), 1)
    for a in actions:
        if a not in node.children:
            node.children[a] = Node(chain=node.chain + (a,), prior_P=p_uniform)
    node.expanded = True

def simulate_one(root, question, max_depth):
    """One MCTS rollout: select to leaf, expand, score, backup."""
    path = [root]
    node = root
    while node.expanded and node.children and len(node.chain) < max_depth:
        a = select(node); node = node.children[a]; path.append(node)
    if not node.expanded and len(node.chain) < max_depth:
        expand(node, question)
        if node.children:
            a = random.choice(list(node.children.keys()))
            node = node.children[a]; path.append(node)
    score = prm_score(question, list(node.chain))
    for n in path: n.N += 1; n.W += score
    return score

def mcts_search(question, gt_len):
    root = Node()
    max_depth = max(2, int(math.ceil(MAX_DEPTH_FACTOR * gt_len)))
    expand(root, question)
    last_high = 0
    for sim in range(N_SIMS):
        s = simulate_one(root, question, max_depth)
        if s >= PRM_HIGH:
            last_high += 1
            if last_high >= 2: break
        else: last_high = 0
    # extract best chain by greedy max-N descent
    node, chain = root, []
    while node.children and len(chain) < max_depth:
        a = max(node.children.items(), key=lambda x: x[1].N)[0]
        chain.append(a); node = node.children[a]
        if prm_score(question, chain) >= PRM_HIGH and len(chain) >= max(2, gt_len-1): break
    return chain, prm_score(question, chain)

print('MCTS+PUCT search ready.')

In [ ]:
# ========== 5. evaluation on §9E.1 L2 ==========
def chain_em(pred, gt):
    if len(pred) != len(gt): return 0.0
    return float(all(p.strip().lower() == g.strip().lower() for p, g in zip(pred, gt)))

def chain_f1(pred, gt):
    pset = set(p.strip().lower() for p in pred)
    gset = set(g.strip().lower() for g in gt)
    if not pset or not gset: return 0.0
    inter = pset & gset
    if not inter: return 0.0
    prec = len(inter) / len(pset); rec = len(inter) / len(gset)
    return 2 * prec * rec / (prec + rec)

results = []
t0 = time.time()
for i, entry in enumerate(L2):
    qid = entry['qid']
    q   = entry['question_text']
    gt  = gt_chain(entry)
    if not gt: continue
    chain, conf = mcts_search(q, len(gt))
    em = chain_em(chain, gt); f1 = chain_f1(chain, gt)
    results.append({'qid': qid, 'gt_len': len(gt), 'pred_len': len(chain),
                     'em': em, 'f1': f1, 'prm_conf': conf})
    if (i+1) % 5 == 0:
        elapsed = time.time() - t0
        print(f'  [{i+1}/{len(L2)}] EM_avg={np.mean([r["em"] for r in results]):.3f}  '
              f'F1_avg={np.mean([r["f1"] for r in results]):.3f}  ({elapsed:.0f}s)')

EM = np.mean([r['em'] for r in results])
F1 = np.mean([r['f1'] for r in results])
print(f'\n>>> §9E.1 L2 (n={len(results)})  EM={EM:.3f}  F1={F1:.3f}  '
      f'(targets EM≥0.55, F1≥0.70)')
print(f'    PASS: EM={EM>=0.55}  F1={F1>=0.70}')

(LOGS / 'B2_mcts_results.jsonl').write_text(
    '\n'.join(json.dumps(r, ensure_ascii=False) for r in results), encoding='utf-8')
(LOGS / 'B2_summary.json').write_text(json.dumps({
    'n': len(results), 'em': EM, 'f1': F1,
    'sims_per_q': N_SIMS, 'k_branch': K_BRANCH, 'c_puct': C_PUCT,
    'pass_em': EM >= 0.55, 'pass_f1': F1 >= 0.70,
}, indent=2), encoding='utf-8')
print(f'wrote logs/B2_mcts_results.jsonl + B2_summary.json')

## Go / No-Go

**PASS:** EM ≥ 0.55 AND F1 ≥ 0.70 on §9E.1 L2 (60 题)

**On EM fail (< 0.55) — recovery ladder:**
1. Bump `N_SIMS` 24 → 64 (more search budget)
2. Bump `K_BRANCH` 6 → 10 (wider candidates per node)
3. Lower `C_PUCT` 1.5 → 0.8 (more exploitation, less exploration)
4. Replace uniform `prior_P` with proposer-confidence (DeepSeek logprobs)
5. Last resort: oracle-conditioned proposer (give DeepSeek the §9E.1 schema as context)

**On F1 fail but EM pass:** chain length is wrong but content is right — relax `MAX_DEPTH_FACTOR` 1.5 → 2.0 and add early-stop reward shaping toward `gt_len` (W4).

**On both fail catastrophically (EM ≤ 0.1):** PRM is mis-aligned with §9E.1 chain semantics. Roll back to W4 plan: re-train PRM on §9E.1 L2 trajectories specifically (currently trained on §9A.3 only).

**Commit once green:**
```
git add nb/B2_mcts_runtime.ipynb logs/B2_summary.json
git commit -m 'B2: L2 MCTS+PUCT runtime — EM=X.XX F1=X.XX on §9E.1 L2'
git tag w3-track-b-mcts
```